In [ ]:
# Scientific computing.
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import json
import IPython
import random

# Resolve Utilities path from the actual notebook location.
# VSCode is fighting the entire time. Try/except for a failsafe. 
# Uses VS Code's injected variable to find the notebook file reliably.

# Ugh. https://github.com/microsoft/vscode-jupyter/issues/8475
try:
    notebook_path = Path(IPython.extract_module_locals()[1]["__vsc_ipynb_file__"])
    utilities_path = notebook_path.parent.parent / "Utilities"
except KeyError:
    # Fallback if run outside VS Code.
    utilities_path = Path().resolve() / "Notebooks" / "Utilities"

sys.path.append(str(utilities_path))

from SI_Utilities import (
    load_survey_data,
    load_tfidf_vectorizer,
    get_agreement_index,
    prepare_model_data_tfidf,
    Leader_R_Cols, 
    Primary_Pairs,
    build_summary_tfidf_df,
)

from Model_Utilities import (
    run_tfidf_nb, 
    split_and_train_tfidf_nb,
    run_tfidf_nb

)

# Machine learning.
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import EasyEnsembleClassifier

In [2]:
# Anchor to the notebook's location to not hardcode paths.
Notebook_Dir = Path().resolve()
Project_Root = Notebook_Dir.parents[1]
Data_Dir = Project_Root / "Clean_Data_Resources"

# Load the parquet from Text_Parsing.ipynb.
Survey_df = pd.read_parquet(Data_Dir / "Survey_df_Text_Parsed.parquet")

# Load scale guide that maps each column/semester/discipline to its scale type.
Likert_Guide_df = pd.read_csv(Data_Dir / "Survey_Results_Likert_Guide.csv")

# Load column metadata.
with open(Data_Dir / "column_metadata.json", "r") as f:
    Column_Metadata = json.load(f)

# Compute averaged leader score.
Survey_df["R_Leader_Average_encoded"] = Survey_df[Leader_R_Cols].mean(axis=1)

In [12]:
# Confirm import and data loaded correctly.
# Print count of working agreement rows. 
test_idx = get_agreement_index(
    "R_Collaborative_Activities_encoded",
    Survey_df,
    Likert_Guide_df
)
print(f"Agreement rows for R_Collaborative_Activities: {len(test_idx)}")

Agreement rows for R_Collaborative_Activities: 31128


In [ ]:
Agreement_df = Survey_df.loc[test_idx]

# Row counts.
print(f"Total survey responses: {len(Survey_df)}")

# Semester and year counts.
print("\nSemester counts within total dataset.")
print(Agreement_df["Semester"].value_counts().to_string())

print("\nYear counts within total dataset.")
print(Agreement_df["Year"].value_counts().sort_index().to_string())

# Counts of content within kept T_ columns.
print("\nT_ column counts within total dataset.")
Text_Cols = [
    "T_Collaboration_Understanding",
    "T_Leader_Performance_Suggestions",
    "T_Other_Suggestions",
    "T_Program_Overall_Suggestions",
]
for col in Text_Cols:
    count = Agreement_df[col].notna().sum()
    print(f"{col.replace('T_', '')}: {count}")

# Overall scale distribution across the full dataset.
print("\nScale distribution across all rows:")
print(f"Agreement scale: {len(test_idx)}")
print(f"Helpfulness scale: {len(Survey_df) - len(test_idx)}")
print(f"Total: {len(Survey_df)}")
print()

# Class balance per R_ column after binarization.
print("\nClass balance per rating column (Agreement scale only, threes dropped):")
R_Cols = [
    "R_Collaborative_Activities_encoded",
    "R_Leader_Helps_Understanding_encoded",
    "R_Leader_Subject_Competence_encoded",
    "R_Leader_Has_Plan_encoded",
    "R_Leader_Willing_To_Help_encoded",
    "R_Overall_Session_Helpfulness_encoded",
]
for col in R_Cols:
    subset = Agreement_df[col].dropna()
    subset = subset[subset != 3]
    positive = (subset > 3).sum()
    negative = (subset < 3).sum()
    total = len(subset)
    print(f"{col.replace('_encoded', '').replace('R_', '')}: "
          f"positive={positive} ({round(positive/total*100, 1)}%) "
          f"negative={negative} ({round(negative/total*100, 1)}%)")
    


Semester counts within total dataset.
Semester
Fall      18293
Spring    12835

Year counts within total dataset.
Year
2019    6921
2020    4346
2021    2432
2022    3881
2023    6603
2024    3267
2025    3678

T_ column counts within total dataset.
Collaboration_Understanding: 26841
Leader_Performance_Suggestions: 24718
Other_Suggestions: 1423
Program_Overall_Suggestions: 21713

Scale distribution across all rows:
Agreement scale: 31128
Helpfulness scale: 3384
Total: 34512


Class balance per rating column (Agreement scale only, threes dropped):
Collaborative_Activities: positive=27738 (96.2%) negative=1103 (3.8%)
Leader_Helps_Understanding: positive=27320 (94.7%) negative=1531 (5.3%)
Leader_Subject_Competence: positive=29116 (97.8%) negative=669 (2.2%)
Leader_Has_Plan: positive=29244 (98.1%) negative=552 (1.9%)
Leader_Willing_To_Help: positive=28890 (97.9%) negative=631 (2.1%)
Overall_Session_Helpfulness: positive=27249 (94.0%) negative=1745 (6.0%)


In [14]:
# Join lemmas back to strings for TfidfVectorizer input.
# TfidfVectorizer expects raw strings, not pre-tokenized lists.
Text_Cols = list(set(t_col for t_col, _ in Primary_Pairs))
for col in Text_Cols:
    Survey_df[col + "_lemma_str"] = Survey_df[col + "_lemmas"].apply(
        lambda x: " ".join(x) if len(x) > 0 else ""
    )

In [15]:
# Pre-load all vectorizers once to avoid repeated disk reads.
Text_Cols = list(set(t_col for t_col, _ in Primary_Pairs))
Vectorizers = {col: load_tfidf_vectorizer(col, Project_Root) for col in Text_Cols}

In [16]:
# Run Naive Bayes with TF-IDF features across all primary pairings.
TF_IDF_NB_Results = []

for t_col, r_col in Primary_Pairs:
    label = f"{t_col.replace('T_', '')} to {r_col.replace('_encoded', '').replace('R_', '')}."
    print(f"Running: {label}")
    result = run_tfidf_nb(
        t_col, r_col, label, Survey_df, Likert_Guide_df,
        Project_Root, vectorizer=Vectorizers[t_col]
    )
    if result:
        TF_IDF_NB_Results.append(result)
        print(f"Accuracy: {result['Accuracy']}. ROC_AUC: {result['ROC_AUC']}. MCC: {result['MCC']}")
    print()

Running: Collaboration_Understanding to Collaborative_Activities.
Accuracy: 0.968. ROC_AUC: 0.776. MCC: 0.0

Running: Leader_Performance_Suggestions to Leader_Helps_Understanding.
Accuracy: 0.942. ROC_AUC: 0.814. MCC: 0.042

Running: Leader_Performance_Suggestions to Leader_Subject_Competence.
Accuracy: 0.976. ROC_AUC: 0.812. MCC: 0.0

Running: Leader_Performance_Suggestions to Leader_Has_Plan.
Accuracy: 0.981. ROC_AUC: 0.766. MCC: 0.0

Running: Leader_Performance_Suggestions to Leader_Willing_To_Help.
Accuracy: 0.978. ROC_AUC: 0.768. MCC: 0.0

Running: Leader_Performance_Suggestions to Leader_Average.
Accuracy: 0.973. ROC_AUC: 0.815. MCC: 0.0

Running: Other_Suggestions to Overall_Session_Helpfulness.
Accuracy: 0.886. ROC_AUC: 0.738. MCC: 0.142

Running: Program_Overall_Suggestions to Overall_Session_Helpfulness.
Accuracy: 0.936. ROC_AUC: 0.74. MCC: 0.0



In [17]:
# Flatten results into a summary table.
TF_IDF_NB_Summary_df = build_summary_tfidf_df(TF_IDF_NB_Results)
print(TF_IDF_NB_Summary_df.reset_index().to_string(index=False))

 index                                                       Pairing     N  Pos %  Neg %  Accuracy    F1  Precision  Recall  ROC AUC   MCC
     0      Collaboration_Understanding to Collaborative_Activities. 24990   96.8    3.2     0.968 0.984      0.968   1.000    0.776 0.000
     1 Leader_Performance_Suggestions to Leader_Helps_Understanding. 20371   94.2    5.8     0.942 0.970      0.942   1.000    0.814 0.042
     2  Leader_Performance_Suggestions to Leader_Subject_Competence. 21041   97.6    2.4     0.976 0.988      0.976   1.000    0.812 0.000
     3            Leader_Performance_Suggestions to Leader_Has_Plan. 21076   98.1    1.9     0.981 0.991      0.981   1.000    0.766 0.000
     4     Leader_Performance_Suggestions to Leader_Willing_To_Help. 20837   97.8    2.2     0.978 0.989      0.978   1.000    0.768 0.000
     5             Leader_Performance_Suggestions to Leader_Average. 21526   97.3    2.7     0.973 0.986      0.973   1.000    0.815 0.000
     6             Other_Su

In [48]:
# Extract top predictive phrases per pairing using likelihood ratios.
# Positive class (1) = phrases most predictive of HIGH ratings.
# Negative class (0) = phrases most predictive of LOW ratings.
from SI_Utilities import get_top_features_by_ratio

for result in TF_IDF_NB_Results:
    vectorizer = result["Vectorizer"]
    nb_model = result["Model"]
    vocab = vectorizer.get_feature_names_out()

    top_features = get_top_features_by_ratio(nb_model, vocab, n=15)

    print(f"{result['Label']}")
    print(f"  Top phrases predicting LOW ratings:")
    for phrase in top_features[0]:
        print(f"    {phrase}")
    print(f"  Top phrases predicting HIGH ratings:")
    for phrase in top_features[1]:
        print(f"    {phrase}")
    print()

Collaboration_Understanding to Collaborative_Activities.
  Top phrases predicting LOW ratings:
    not work collaboratively
    not collaborative
    not collaborative work
    not worked collaboratively
    negative
    tables
    never worked collaboratively
    not collaborated
    work very collaboratively
    worse
    not collaborating
    never collaborative
    worked collaboratively
    students able explain
    ochem
  Top phrases predicting HIGH ratings:
    perspectives
    topic
    helps
    better understand
    new
    helped lot
    good
    ways
    better
    ideas
    allowed
    understand better
    well
    information
    able

Leader_Performance_Suggestions to Leader_Helps_Understanding.
  Top phrases predicting LOW ratings:
    requirement
    know content
    instructor not
    wasted time
    hate
    explain not
    instead telling
    force people
    not learning
    more preparation
    leave more confused
    problems more depth
    actually teach
    w

In [18]:
# Test different thresholds to find where the model actually discriminates.
sample_responses = [
    "This was the best SI session I have attended, everything clicked.",
    "The leader had no idea what they were doing and wasted my time.",
    "Pretty good session, helped me understand the main concepts.",
    "I left more confused than when I arrived, very disorganized.",
    "Nothing special, just went through the homework problems.",
    "Absolutely loved it, the practice problems were perfect for the exam.",
    "The leader rushed through everything and did not explain anything.",
    "It was okay, nothing I could not have done on my own.",
    "Could not follow along at all, the leader was unprepared.",
    "Great energy, made a difficult topic actually make sense.",
]

thresholds = [0.5, 0.3, 0.2, 0.1, 0.05]

result = TF_IDF_NB_Results[0]
vectorizer = result["Vectorizer"]
nb_model = result["Model"]

for response in sample_responses:
    vec = vectorizer.transform([response])
    prob = nb_model.predict_proba(vec)[0][1]
    print(f"Response: {response[:60]}")
    for t in thresholds:
        prediction = "POSITIVE" if prob > t else "NEGATIVE"
        print(f"  threshold={t}: {prediction} | P(pos)={round(prob, 3)}")
    print()

Response: This was the best SI session I have attended, everything cli
  threshold=0.5: POSITIVE | P(pos)=0.95
  threshold=0.3: POSITIVE | P(pos)=0.95
  threshold=0.2: POSITIVE | P(pos)=0.95
  threshold=0.1: POSITIVE | P(pos)=0.95
  threshold=0.05: POSITIVE | P(pos)=0.95

Response: The leader had no idea what they were doing and wasted my ti
  threshold=0.5: POSITIVE | P(pos)=0.954
  threshold=0.3: POSITIVE | P(pos)=0.954
  threshold=0.2: POSITIVE | P(pos)=0.954
  threshold=0.1: POSITIVE | P(pos)=0.954
  threshold=0.05: POSITIVE | P(pos)=0.954

Response: Pretty good session, helped me understand the main concepts.
  threshold=0.5: POSITIVE | P(pos)=0.992
  threshold=0.3: POSITIVE | P(pos)=0.992
  threshold=0.2: POSITIVE | P(pos)=0.992
  threshold=0.1: POSITIVE | P(pos)=0.992
  threshold=0.05: POSITIVE | P(pos)=0.992

Response: I left more confused than when I arrived, very disorganized.
  threshold=0.5: POSITIVE | P(pos)=0.983
  threshold=0.3: POSITIVE | P(pos)=0.983
  threshold=0.2: P

In [19]:
# Quick test of EasyEnsemble on first pairing.
X, y, vectorizer = prepare_model_data_tfidf(
    "T_Collaboration_Understanding",
    "R_Collaborative_Activities_encoded",
    Survey_df,
    Likert_Guide_df,
    Project_Root,
    vectorizer=Vectorizers["T_Collaboration_Understanding"]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2026, stratify=y
)

for n in [10, 20, 30, 50, 100, 200]:
    ee = EasyEnsembleClassifier(n_estimators=n, random_state=2026)
    ee.fit(X_train, y_train)
    print(f"n_estimators={n}")
    for response in sample_responses:
        vec = vectorizer.transform([response])
        prob = ee.predict_proba(vec)[0][1]
        prediction = "POSITIVE" if prob > 0.5 else "NEGATIVE"
        print(f"  {response[:65]}: {prediction} | P(pos): {round(prob, 3)}")
    print()

n_estimators=10
  This was the best SI session I have attended, everything clicked.: NEGATIVE | P(pos): 0.497
  The leader had no idea what they were doing and wasted my time.: NEGATIVE | P(pos): 0.497
  Pretty good session, helped me understand the main concepts.: POSITIVE | P(pos): 0.549
  I left more confused than when I arrived, very disorganized.: NEGATIVE | P(pos): 0.497
  Nothing special, just went through the homework problems.: NEGATIVE | P(pos): 0.497
  Absolutely loved it, the practice problems were perfect for the e: NEGATIVE | P(pos): 0.497
  The leader rushed through everything and did not explain anything: NEGATIVE | P(pos): 0.392
  It was okay, nothing I could not have done on my own.: NEGATIVE | P(pos): 0.392
  Could not follow along at all, the leader was unprepared.: NEGATIVE | P(pos): 0.39
  Great energy, made a difficult topic actually make sense.: POSITIVE | P(pos): 0.5

n_estimators=20
  This was the best SI session I have attended, everything clicked.: NEGATIVE 

In [20]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
import numpy as np

# Get the TF-IDF matrix for negative examples only.
agreement_idx = get_agreement_index(
    "R_Collaborative_Activities_encoded",
    Survey_df,
    Likert_Guide_df
)
subset = Survey_df.loc[agreement_idx].copy()
subset = subset.dropna(subset=["R_Collaborative_Activities_encoded"])
subset = subset[subset["T_Collaboration_Understanding_lemma_str"].str.strip() != ""]
subset = subset[subset["R_Collaborative_Activities_encoded"] != 3]
y = (subset["R_Collaborative_Activities_encoded"] > 3).astype(int)

# Separate negative examples.
neg_mask = y == 0
neg_strings = subset.loc[neg_mask, "T_Collaboration_Understanding_lemma_str"].tolist()
neg_X = Vectorizers["T_Collaboration_Understanding"].transform(neg_strings)

# Normalize for better clustering on sparse TF-IDF vectors.
neg_X_norm = normalize(neg_X)

# Fit KMeans -- try 5 clusters based on expected subconcepts.
kmeans = KMeans(n_clusters=5, random_state=2026, n_init=10)
cluster_labels = kmeans.fit_predict(neg_X_norm)

# Show top terms per cluster to see if subconcepts emerge.
vocab = Vectorizers["T_Collaboration_Understanding"].get_feature_names_out()
print(f"Total negative examples: {len(neg_strings)}")
print()

for cluster_id in range(5):
    cluster_mask = cluster_labels == cluster_id
    cluster_size = cluster_mask.sum()
    # Average TF-IDF scores for this cluster.
    cluster_center = neg_X_norm[cluster_mask].mean(axis=0)
    top_idx = np.asarray(cluster_center).flatten().argsort()[-10:][::-1]
    top_terms = [vocab[i] for i in top_idx]
    # Sample a real response from this cluster.
    sample = neg_strings[np.where(cluster_mask)[0][0]]
    print(f"Cluster {cluster_id} ({cluster_size} examples)")
    print(f"  Top terms: {top_terms}")
    print(f"  Sample: {sample[:120]}")
    print()

Total negative examples: 812

Cluster 0 (92 examples)
  Top terms: ['work collaboratively', 'collaboratively', 'work', 'not work collaboratively', 'not work', 'worked collaboratively', 'worked', 'not', 'work together', 'never']
  Sample: work collaboratively feel beneficial

Cluster 1 (74 examples)
  Top terms: ['understanding', 'contributed', 'not', 'understanding concepts', 'material', 'classmates', 'working', 'better', 'better understanding', 'concepts']
  Sample: found help understanding concepts

Cluster 2 (46 examples)
  Top terms: ['collaborative', 'collaborative work', 'work', 'not collaborative', 'collaborative activities', 'not collaborative work', 'not', 'activities', 'never collaborative', 'never']
  Sample: never collaborative work mainly individual

Cluster 3 (534 examples)
  Top terms: ['helped', 'work', 'not', 'understand', 'help', 'helpful', 'collaborate', 'concepts', 'problems', 'more']
  Sample: talking things very helpful

Cluster 4 (66 examples)
  Top terms: ['not'

In [21]:
# Show real response examples from each cluster with their paired ratings.
neg_responses = subset.loc[neg_mask, "T_Collaboration_Understanding"].tolist()
neg_ratings = subset.loc[neg_mask, "R_Collaborative_Activities_encoded"].tolist()

for cluster_id in range(5):
    cluster_mask = cluster_labels == cluster_id
    cluster_indices = np.where(cluster_mask)[0]
    cluster_size = len(cluster_indices)
    sample_indices = cluster_indices[:10]

    print(f"Cluster {cluster_id} ({cluster_size} examples)")
    for idx in sample_indices:
        print(f"  [{neg_ratings[idx]}] {neg_responses[idx][:150]}")
    print()

Cluster 0 (92 examples)
  [1.0] We don't work collaboratively often, but when we do I feel that it is beneficial. 
  [2.0] We basically never work collaboratively. 
  [1.0] I wouldn't say I have worked collaboratively.
  [1.0] We don't work collaboratively.
  [1.0] We don’t work together. 
  [1.0] we don't work collaboratively.
  [1.0] We do not really work collaboratively in this SI.
  [2.0] We do not work collaboratively 
  [2.0] In my ECON 2013 SI session, I really never work collaboratively with other students. The most collaborative thing we do is a worksheet with practice p
  [1.0] We don’t ever work collaboratively 

Cluster 1 (74 examples)
  [2.0] I found SI didn’t help me at all understanding concepts
  [1.0] I believe that being able to talk to others and explain the concepts to one another is helpful, however, this does not happen often. I have not notice
  [2.0] I have attended most of the SI sessions, and it has not impacted my understanding since we do not do it often.
  

In [ ]:
# Pick a random real response and show its TF-IDF breakdown.

vectorizer = Vectorizers["T_Collaboration_Understanding"]
vocab = vectorizer.get_feature_names_out()

# Pick a random non-empty response.
non_empty = Survey_df[Survey_df["T_Collaboration_Understanding"].notna()]
random_row = non_empty.sample(1, random_state=random.randint(0, 9999)).iloc[0]

response = random_row["T_Collaboration_Understanding"]
rating = random_row["R_Collaborative_Activities_encoded"]

vec = vectorizer.transform([response])
scores = vec.toarray()[0]
top_idx = scores.argsort()[-10:][::-1]
top_terms = [(vocab[i], round(scores[i], 3)) for i in top_idx if scores[i] > 0]

print(f"Response: {response}")
print(f"Rating: {rating}")
print(f"Most distinctive phrases:")
for term, score in top_terms:
    print(f"  '{term}' score: {score}")

Response: It helps me when we are solving problems and I'm blanking on how to do something. 
Rating: 5.0
Most distinctive phrases:
  'it' score: 0.716
  'solving problems' score: 0.475
  'solving' score: 0.413
  'problems' score: 0.227
  'helps' score: 0.198
